In [ ]:
# Kurulum: Colab'da repoyu indirir, yerelde hiçbir şey yapmaz.
import os, sys
REPO = "https://github.com/mehmetalibeylikci/python-cevresel-modelleme"
if "google.colab" in sys.modules:
    if not os.path.exists("/content/python-cevresel-modelleme"):
        !git clone -q {REPO} /content/python-cevresel-modelleme
    %cd /content/python-cevresel-modelleme
    %pip install -q sweetviz
else:
    while not os.path.exists("src/model.py") and os.getcwd() != "/":
        os.chdir("..")
sys.path.insert(0, os.getcwd())

# Gün 3 — Görselleştirme ve Senaryolar

**İYTE Çevre Topluluğu · Python ile Çevresel Modelleme**

Dün modelimizi kurduk ve önemli bir şey öğrendik: model **mutlak değerleri** iyi tahmin edemiyor, çünkü parametreler belirsiz. Ama bir modelin en güçlü kullanımı çoğu zaman **"ya şöyle olursa?"** sorusudur: aynı model, farklı koşullar altında nasıl davranıyor?

Bugün senaryolar kuracak ve sonuçları **anlaşılır, dürüst ve sunuma hazır** grafiklerle anlatacağız.

**Öğrenme hedefleri**
- matplotlib'in yapısını (`fig`, `ax`) anlamak
- Kötü bir grafiği adım adım iyileştirmek
- Senaryo analizi yapmak ve sonuçları bir metrikle özetlemek
- Çok panelli figür, ısı haritası ve interaktif grafik üretmek

| Süre | Konu |
|---|---|
| 0–30 | matplotlib anatomisi ve kötü bir grafiği düzeltmek |
| 30–70 | Senaryolar |
| 70–100 | Sunuma hazır figürler |
| 100–120 | Kapanış: modelin sınırları ve sonraki adımlar |

In [ ]:
from dataclasses import replace

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.model import S0_DATE, S0_HM3, Params, active_fill_pct, simulate

monthly = pd.read_csv("data/processed/tahtali_monthly.csv")
monthly.index = pd.to_datetime(monthly.pop("month"))
data = monthly.loc[S0_DATE:]

base = Params()
baseline = simulate(data["precipitation_mm"], data["et0_mm"], S0_HM3, base)
baseline.head()

## 1. matplotlib anatomisi

matplotlib'de her grafik iki nesneden oluşur:
- **`fig`** (Figure): kâğıdın kendisi. Boyutu, kaydetme işlemi buradan yönetilir.
- **`ax`** (Axes): kâğıdın üzerindeki bir grafik alanı. Çizgiler, eksenler, başlık buraya aittir.

`fig, ax = plt.subplots()` ikisini birlikte oluşturur. Bir figürde birden fazla `ax` olabilir: `plt.subplots(3, 1)`.

### Kötü bir grafik

Aşağıdaki grafik "çalışıyor", ama bir sunumda gösterseniz kimse bir şey anlamaz. Neden?

In [ ]:
plt.plot(baseline)
plt.show()

Sorunlar:
1. Birbiriyle ilgisiz büyüklükler (depolama ve aylık akışlar) aynı eksende.
2. Başlık yok, eksen etiketi yok, **birim** yok.
3. Hangi çizginin ne olduğu belli değil.
4. Okuyucuya "neye bakmalıyım?" sorusunun cevabı verilmiyor.

Adım adım düzeltelim.

In [ ]:
fill = active_fill_pct(baseline["storage"])

fig, ax = plt.subplots(figsize=(10, 3.5))                       # 1) boyutu belirle, tek bir büyüklük seç
ax.plot(fill.index, fill, color="tab:blue", label="Model")      # 2) anlamlı bir büyüklük: aktif doluluk
ax.set_title("Tahtalı Barajı aktif doluluk, temel senaryo")    # 3) başlık
ax.set_ylabel("Aktif doluluk (%)")                              # 4) birimli eksen etiketi
ax.set_ylim(0, 105)                                             # 5) dürüst eksen: 0'dan başla
ax.axhline(10, color="gray", linestyle="--", linewidth=1, label="Kritik seviye (%10)")  # 6) bağlam
ax.legend(loc="lower left")
plt.show()

> 💡 **Altın kural:** Her grafik tek bir soruya cevap vermeli. Bu grafik "Baraj ne kadar dolu ve kritik seviyeye ne kadar yakın?" sorusunu cevaplıyor.

### 🟢 Alıştırma 1
Aylık **giren su** (`inflow`) ve **buharlaşmayı** (`evaporation`) aynı grafikte gösterin. Başlık, birimli eksen etiketi ve lejant eklemeyi unutmayın.

In [ ]:
# BEGIN SOLUTION
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(baseline.index, baseline["inflow"], color="tab:blue", label="Giren su")
ax.plot(baseline.index, baseline["evaporation"], color="tab:orange", label="Buharlaşma")
ax.set_title("Tahtalı: aylık giren su ve buharlaşma")
ax.set_ylabel("Hacim (hm³/ay)")
ax.legend()
plt.show()
# END SOLUTION

## 2. Senaryolar

İklim değişikliği projeksiyonları, Akdeniz havzasında **daha az yağış** ve **daha yüksek sıcaklık** (yani daha fazla buharlaşma) öngörüyor. Nüfus artışı da **talebi** artırıyor. Bu üç etkiyi ayrı ayrı deneyelim:

| Senaryo | Değişiklik |
|---|---|
| Temel | — |
| Kurak | yağış × 0,8 (−%20) |
| Sıcak | ET0 × 1,1 (+%10) |
| Yüksek talep | çekim × 1,15 (+%15) |

Senaryolar, aynı modele farklı girdiler ve parametreler vermekten ibaret. Her birini bir sözlükte tanımlayalım:

In [ ]:
scenarios = {
    "Temel": dict(precip_factor=1.0, et0_factor=1.0, params=base),
    "Kurak (yağış −%20)": dict(precip_factor=0.8, et0_factor=1.0, params=base),
    "Sıcak (ET0 +%10)": dict(precip_factor=1.0, et0_factor=1.1, params=base),
    "Yüksek talep (+%15)": dict(precip_factor=1.0, et0_factor=1.0, params=replace(base, withdrawal=base.withdrawal * 1.15)),
}


def run_scenario(precip_factor, et0_factor, params):
    return simulate(data["precipitation_mm"] * precip_factor, data["et0_mm"] * et0_factor, S0_HM3, params)


results = {name: run_scenario(**spec) for name, spec in scenarios.items()}
results["Kurak (yağış −%20)"].head()

Sonuçları tek bir tabloda özetleyelim. **Kritik seviyenin altında geçen ay sayısı** iyi bir metrik: hem anlaşılır hem de şehir için doğrudan bir anlamı var.

In [ ]:
summary = pd.DataFrame({
    name: {
        "Kritik altı ay sayısı": int((active_fill_pct(out["storage"]) < 10).sum()),
        "En düşük doluluk (%)": active_fill_pct(out["storage"]).min(),
        "Karşılanamayan çekim (hm³)": out["shortfall"].sum(),
        "Taşan su (hm³)": out["spill"].sum(),
    }
    for name, out in results.items()
}).T
summary.round(1)

### 🟢 Alıştırma 2
Tabloyu yorumlayın: Hangi değişiklik baraj için en tehlikeli? Yağıştaki %20'lik düşüş ile talepteki %15'lik artışı karşılaştırın. Sonra **birleşik** bir senaryo ekleyin: kurak **ve** sıcak **ve** yüksek talep, hepsi birlikte. Senaryoya `"Hepsi birlikte"` adını verin, `results` sözlüğüne ekleyin ve tabloya ekleyip yeniden gösterin.

In [ ]:
# BEGIN SOLUTION
scenarios["Hepsi birlikte"] = dict(precip_factor=0.8, et0_factor=1.1, params=replace(base, withdrawal=base.withdrawal * 1.15))
results["Hepsi birlikte"] = run_scenario(**scenarios["Hepsi birlikte"])
summary.loc["Hepsi birlikte"] = {
    "Kritik altı ay sayısı": int((active_fill_pct(results["Hepsi birlikte"]["storage"]) < 10).sum()),
    "En düşük doluluk (%)": active_fill_pct(results["Hepsi birlikte"]["storage"]).min(),
    "Karşılanamayan çekim (hm³)": results["Hepsi birlikte"]["shortfall"].sum(),
    "Taşan su (hm³)": results["Hepsi birlikte"]["spill"].sum(),
}
print(summary.round(1))
# END SOLUTION

## 3. Sunuma hazır figürler

### 3.1 Çok panelli figür

Bir figürde birden fazla panel kullanarak bir **hikâye** anlatabiliriz: (a) baraj ne durumda, (b) neden, (c) senaryolar ne diyor. `sharex=True` ile tüm paneller aynı zaman eksenini paylaşır.

In [ ]:
colors = {
    "Temel": "black",
    "Kurak (yağış −%20)": "tab:red",
    "Sıcak (ET0 +%10)": "tab:orange",
    "Yüksek talep (+%15)": "tab:purple",
    "Hepsi birlikte": "darkred",
}

fig, axes = plt.subplots(3, 1, figsize=(11, 9), sharex=True, gridspec_kw={"height_ratios": [2, 1.4, 2]})

# (a) Temel senaryoda depolama
ax = axes[0]
ax.fill_between(baseline.index, base.s_min, baseline["storage"], color="tab:blue", alpha=0.3)
ax.plot(baseline.index, baseline["storage"], color="tab:blue")
ax.axhline(base.s_max, color="gray", linewidth=1)
ax.axhline(base.s_min, color="gray", linewidth=1)
ax.text(baseline.index[0], base.s_max + 5, "Normal kot (306,65 hm³)", fontsize=8, color="gray")
ax.text(baseline.index[0], base.s_min + 5, "Ölü hacim (19,6 hm³)", fontsize=8, color="gray")
ax.set_ylabel("Depolama (hm³)")
ax.set_ylim(0, 340)
ax.set_title("(a) Temel senaryoda göldeki su")

# (b) Aylık bütçe: giren su yukarı, çıkanlar aşağı
ax = axes[1]
ax.bar(baseline.index, baseline["inflow"], width=25, color="tab:blue", label="Giren su")
ax.bar(baseline.index, -baseline["evaporation"], width=25, color="tab:orange", label="Buharlaşma")
ax.bar(baseline.index, -baseline["withdrawal"], width=25, bottom=-baseline["evaporation"], color="tab:gray", label="Çekim")
ax.axhline(0, color="black", linewidth=0.8)
ax.set_ylabel("hm³/ay")
ax.set_title("(b) Aylık su bütçesi: giren (+) ve çıkan (−)")
ax.legend(loc="upper left", ncol=3, fontsize=8)

# (c) Senaryolar
ax = axes[2]
for name, out in results.items():
    ax.plot(out.index, active_fill_pct(out["storage"]), color=colors.get(name, "darkred"), label=name,
            linewidth=2 if name == "Temel" else 1.3)
ax.axhline(10, color="gray", linestyle="--", linewidth=1)
ax.text(baseline.index[-1], 12, "kritik seviye", fontsize=8, color="gray", ha="right")
ax.set_ylim(0, 105)
ax.set_ylabel("Aktif doluluk (%)")
ax.set_title("(c) Senaryolar")
ax.legend(loc="lower left", fontsize=8)

fig.suptitle("Tahtalı Barajı su bütçesi modeli, 2019–2026", fontsize=14, fontweight="bold")
fig.tight_layout()

import os

os.makedirs("figures", exist_ok=True)
fig.savefig("figures/tahtali_senaryolar.png", dpi=200, bbox_inches="tight")
plt.show()

`savefig` ile figürü yüksek çözünürlükte (`dpi=200`) kaydettik. Colab'da soldaki 📁 dosyalar panelinden `figures/` klasörünü açıp indirebilirsiniz.

### 3.2 Mevsimsel ısı haritası

Zaman serisinin hem **mevsimsel** hem **yıllar arası** yapısını aynı anda görmenin güzel bir yolu: satırlar yıllar, sütunlar aylar. **seaborn** bunu tek satırda çizer.

In [ ]:
import seaborn as sns

pct = active_fill_pct(results["Kurak (yağış −%20)"]["storage"])
grid = pct.groupby([pct.index.year, pct.index.month]).mean().unstack()
grid.columns = ["Oca", "Şub", "Mar", "Nis", "May", "Haz", "Tem", "Ağu", "Eyl", "Eki", "Kas", "Ara"]

fig, ax = plt.subplots(figsize=(10, 4))
sns.heatmap(grid, cmap="RdYlBu", vmin=0, vmax=100, annot=True, fmt=".0f", linewidths=0.5,
            cbar_kws={"label": "Aktif doluluk (%)"}, ax=ax)
ax.set_title("Kurak senaryoda (yağış −%20) aylık aktif doluluk")
ax.set_xlabel("")
ax.set_ylabel("Yıl")
ax.tick_params(axis="y", rotation=0)
plt.show()

Renk seçimi bilinçli: kırmızı "tehlike", mavi "su" çağrışımı yapıyor. `vmin=0, vmax=100` ile renk ölçeğini sabitledik. Böylece farklı senaryoların haritaları birbiriyle karşılaştırılabilir.

### 🟢 Alıştırma 3
Aynı ısı haritasını **temel** senaryo için çizin. İki haritayı karşılaştırınca ne görüyorsunuz?

In [ ]:
# BEGIN SOLUTION
pct = active_fill_pct(results["Temel"]["storage"])
grid = pct.groupby([pct.index.year, pct.index.month]).mean().unstack()
grid.columns = ["Oca", "Şub", "Mar", "Nis", "May", "Haz", "Tem", "Ağu", "Eyl", "Eki", "Kas", "Ara"]
fig, ax = plt.subplots(figsize=(10, 4))
sns.heatmap(grid, cmap="RdYlBu", vmin=0, vmax=100, annot=True, fmt=".0f", linewidths=0.5,
            cbar_kws={"label": "Aktif doluluk (%)"}, ax=ax)
ax.set_title("Temel senaryoda aylık aktif doluluk")
ax.set_xlabel("")
ax.set_ylabel("Yıl")
ax.tick_params(axis="y", rotation=0)
plt.show()
# END SOLUTION

### 3.3 İnteraktif grafik: plotly

Statik grafikler rapor ve sunum içindir. **Keşif** için ise üzerine gelince değerleri gösteren, yakınlaştırılabilen interaktif grafikler çok kullanışlıdır. **plotly** bunu kolaylaştırır. Bir çizginin üzerine gelin, lejantta bir senaryoya tıklayıp gizleyin.

In [ ]:
import plotly.graph_objects as go

fig = go.Figure()
for name, out in results.items():
    fig.add_trace(go.Scatter(x=out.index, y=active_fill_pct(out["storage"]).round(1), name=name,
                             line=dict(color=colors.get(name, "darkred").replace("tab:", ""))))
fig.add_hline(y=10, line_dash="dash", line_color="gray", annotation_text="kritik seviye")
fig.update_layout(title="Tahtalı: senaryolara göre aktif doluluk", yaxis_title="Aktif doluluk (%)",
                  yaxis_range=[0, 105], hovermode="x unified", height=450)
fig

## 4. Kapanış: Bu model neyi söyleyebilir, neyi söyleyemez?

**Söyleyebildikleri**
- Su bütçesinin hangi bileşenlerinin önemli olduğunu: havzadan gelen akış, göle düşen yağmurdan çok daha önemli.
- Senaryoları **göreli olarak** karşılaştırmayı: "Yağıştaki %20'lik düşüş, talepteki %15'lik artıştan daha mı tehlikeli?"
- Sonucun hangi parametrelere hassas olduğunu (Gün 2).

**Söyleyemedikleri**
- Barajın gelecek ay **tam olarak** kaç hm³ olacağını. Parametreler belirsiz ve model 2022–2025 kuraklığını temel parametrelerle yakalayamadı.
- Yağış dağılımının değişmesini: senaryolarımız her ayı aynı oranda azalttı, oysa iklim değişikliği daha uzun kuraklıklar ve daha şiddetli sağanaklar da getirebilir.

**Sıradaki adımlar**
- **Gerçek iklim projeksiyonları:** iklim modellerinin çıktıları genellikle **NetCDF** formatında gelir ve Python'da **xarray** ile okunur. Senaryo çarpanları yerine gerçek projeksiyonlarla çalışmak bu eğitimin doğal devamı.
- **Bonus modül:** `bonus/fizik-mi-veri-mi/` klasöründe aynı problemi bir makine öğrenmesi modeliyle çözüp SHAP ile yorumluyoruz.
- **Kendi verin:** Bu üç günde yaptıklarımız (veri temizleme → kütle dengesi modeli → senaryo ve görselleştirme) bir göl, bir akifer, bir atık su arıtma tesisi ya da bir karbon bütçesi için de aynı şekilde çalışır.

🎉 **Tebrikler, eğitimi tamamladınız!**

---
## Zaman kalırsa

### 🔵 Alıştırma 4: Kendi senaryonuz
Kendi senaryonuzu tasarlayın ve sonucunu grupla paylaşın. Fikirler:
- Kuraklıkta İZSU çekimi %30 azaltırsa? (İpucu: `simulate` sabit çekim kullanır; döngüyü kendiniz yazmanız gerekebilir.)
- Sadece kış yağışları azalırsa? (İpucu: `data.index.month.isin([12, 1, 2])`)

### 🔵 Alıştırma 5: Sonuçlar parametrelere ne kadar bağlı?
Gün 2'de gözlemlerle daha uyumlu bulduğunuz bir $C$ ve $W$ kombinasyonuyla (ör. $C = 0{,}20$, $W = 8$) senaryo tablosunu yeniden üretin. Senaryoların **sıralaması** değişiyor mu? Değişmiyorsa bu, model belirsiz olsa bile göreli sonuçlara güvenebileceğimiz anlamına gelir mi?

In [ ]:
# BEGIN SOLUTION
alt = replace(base, runoff_coef=0.20, withdrawal=8.0)
alt_scenarios = {
    "Temel": (1.0, 1.0, alt),
    "Kurak (yağış −%20)": (0.8, 1.0, alt),
    "Sıcak (ET0 +%10)": (1.0, 1.1, alt),
    "Yüksek talep (+%15)": (1.0, 1.0, replace(alt, withdrawal=alt.withdrawal * 1.15)),
    "Hepsi birlikte": (0.8, 1.1, replace(alt, withdrawal=alt.withdrawal * 1.15)),
}
alt_summary = pd.DataFrame({
    name: {
        "Kritik altı ay (C=0,20, W=8)": int((active_fill_pct(out["storage"]) < 10).sum()),
        "En düşük doluluk % (C=0,20, W=8)": active_fill_pct(out["storage"]).min(),
    }
    for name, out in ((n, run_scenario(p, e, params)) for n, (p, e, params) in alt_scenarios.items())
}).T
print(pd.concat([summary[["Kritik altı ay sayısı", "En düşük doluluk (%)"]], alt_summary], axis=1).round(1))
# END SOLUTION